# ChagaSight — Final Ensemble Evaluation v3.1

## What this notebook produces

| Output | Use |
|--------|-----|
| TPR@5% (official, 10k perms) | Primary thesis metric |
| AUROC, AUPRC with 95% bootstrap CI | Thesis results table |
| MCC, NPV, sensitivity, specificity, F1 | Thesis Table 4.x |
| NNS (Number Needed to Screen) | Clinical interpretation paragraph |
| Per-fold TPR@5% table | Demonstrates ensemble stability |
| Per-dataset breakdown | Shows generalisation across populations |
| 6 publication-quality figures (300 dpi) | Thesis Figures 4.1–4.6 |
| ensemble_predictions.csv | Raw predictions for further analysis |
| FINAL_ENSEMBLE_MODEL.pt | Deployment / demo |

## Paper benchmarks to beat

| System | CV score | Val set score |
|--------|----------|--------------|
| Van Santvliet et al. 2025 (top team) | 0.490 ± 0.008 | 0.445 |
| Kim et al. 2025 | 0.507 | 0.369 |
| Random baseline | 0.05 | — |

## ⚠️  Data-leakage note
Both ST-MEM and MAE pretraining used the **full 366k dataset** before fold splits were
applied to fine-tuning.  The pretrained encoder has therefore "seen" the validation samples
(in an unsupervised context) before fine-tuning evaluation.  This is standard practice
in self-supervised learning (it does not provide label information) but should be disclosed
in the thesis.  To obtain a fully clean estimate, cross-validation would need to exclude
pretraining data for each held-out fold — impractical at this scale.


## Cell 1 — Imports

In [ ]:
import sys, warnings
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    average_precision_score, matthews_corrcoef,
)
warnings.filterwarnings('ignore', category=UserWarning)

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Official PhysioNet metric
helper_path = project_root / 'external' / 'official_2025'
if str(helper_path) not in sys.path:
    sys.path.insert(0, str(helper_path))

OFFICIAL = False
try:
    from helper_code import compute_challenge_score, compute_auc as _compute_auc
    OFFICIAL = True
    print('Official PhysioNet metric: ENABLED')
except ImportError:
    print('Official metric not found — using sklearn approximation')
    print(f'  Expected: {helper_path / "helper_code.py"}')

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## Cell 2 — Configuration

In [ ]:
CHECKPOINT_DIR   = project_root / 'checkpoints'
DATA_DIR         = project_root / 'data' / 'processed'
METADATA_CSV     = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR       = DATA_DIR / '2d_images'
SIGNALS_DIR      = DATA_DIR / '1d_signals_100hz'
FIGURES_DIR      = CHECKPOINT_DIR / 'thesis_figures'
EVAL_CKPT_DIR    = CHECKPOINT_DIR / 'evaluation_checkpoints'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EVAL_CKPT_DIR.mkdir(exist_ok=True)

INFERENCE_BATCH  = 32     # safe for 6 GB GPU in no-grad inference
SAVE_EVERY_N     = 50     # batch-level checkpoint frequency
N_PERMS_FINAL    = 10000  # official permutations
N_PERMS_FOLD     = 5000   # per-fold (faster, still accurate)
N_BOOTSTRAP      = 1000   # CI bootstrap resamples
SEED             = 12345

# Verify all 5 fold checkpoints exist before starting
fold_ckpts = []
for fold in range(5):
    p = CHECKPOINT_DIR / f'fold{fold}_best.pt'
    assert p.exists(), (
        f'Missing fold{fold}_best.pt — train fold {fold} first '
        f'(10_train_fold_FINAL_v10.1.ipynb with FOLD={fold})'
    )
    fold_ckpts.append(p)
    mb = p.stat().st_size / 1e6
    print(f'  fold{fold}_best.pt  {mb:.0f} MB')
print(f'All 5 checkpoints found.')


## Cell 3 — Load All 5 Fold Models

In [ ]:
models          = []
fold_val_scores = []

for fold, ckpt_path in enumerate(fold_ckpts):
    m = HybridChagasModel(
        img_size=(24, 2048), patch_size_2d=(8, 64),
        num_leads=12, seq_len_1d=1000, patch_size_1d=50,
        embed_dim=768, depth=12, num_heads=12,
        use_aol=True, use_demographics=True,
    )
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m.load_state_dict(ckpt['model_state_dict'])
    m.to(device).eval()
    models.append(m)

    vs = ckpt.get('val_score', None)   # None if key absent — do not default to 0
    fold_val_scores.append(vs)
    score_str = f'{vs:.4f}' if vs is not None else 'n/a'
    print(f'Fold {fold}: val_score={score_str}')

total_params = sum(p.numel() for p in models[0].parameters())
print(f'\nModel: HybridChagasModel  |  {total_params:,} params per fold')
valid = [s for s in fold_val_scores if s is not None]
if valid:
    print(f'Fold scores: mean={np.mean(valid):.4f}  std={np.std(valid):.4f}')


## Cell 4 — Ensemble Inference (batch-level checkpointing)

In [ ]:
import time

all_probs, all_labels, all_ids, all_datasets, all_folds_list = [], [], [], [], []
total_start = time.time()

for fold in range(5):
    fold_done = EVAL_CKPT_DIR / f'fold{fold}_complete.npz'

    if fold_done.exists():
        d = np.load(fold_done, allow_pickle=True)
        all_probs.extend(d['probs'].tolist())
        all_labels.extend(d['labels'].tolist())
        all_ids.extend(d['ids'].tolist())
        all_datasets.extend(d['datasets'].tolist())
        all_folds_list.extend([fold] * len(d['labels']))
        print(f'Fold {fold}: loaded from cache  ({len(d["labels"]):,} samples)')
        continue

    print(f'Fold {fold}: running inference ...')
    fold_start = time.time()

    _, val_loader = create_dataloaders(
        metadata_csv=str(METADATA_CSV),
        images_dir=str(IMAGES_DIR),
        signals_dir=str(SIGNALS_DIR),
        fold=fold,
        batch_size=INFERENCE_BATCH,
        num_workers=0,
        use_weighted_sampling=False,
        augment_train=False,
    )

    partial = EVAL_CKPT_DIR / f'fold{fold}_partial.npz'
    start_batch = 0
    fp, fl, fi, fd = [], [], [], []

    if partial.exists():
        d = np.load(partial, allow_pickle=True)
        fp, fl = d['probs'].tolist(), d['labels'].tolist()
        fi, fd = d['ids'].tolist(), d['datasets'].tolist()
        start_batch = int(d['last_batch']) + 1
        print(f'  Resuming from batch {start_batch} ({len(fp):,} samples already done)')

    with torch.no_grad():
        for bi, batch in enumerate(tqdm(val_loader, desc=f'Fold {fold}', leave=False)):
            if bi < start_batch:
                continue
            imgs  = batch['image'].to(device, non_blocking=True)
            sigs  = batch['signal'].to(device, non_blocking=True)
            ages  = batch['age'].to(device, non_blocking=True)
            sexes = batch['sex'].to(device, non_blocking=True)
            hlab  = batch['hard_label'].numpy()

            preds = []
            for m in models:
                out = m(imgs, sigs, ages, sexes)
                preds.append(torch.sigmoid(out['logits']).cpu().numpy())
            ens = np.mean(np.stack(preds), axis=0)

            fp.extend(ens.tolist())
            fl.extend(hlab.tolist())
            fi.extend(batch['id'])
            fd.extend(batch['dataset'])

            if (bi + 1) % SAVE_EVERY_N == 0:
                np.savez(partial, probs=np.array(fp), labels=np.array(fl),
                         ids=fi, datasets=fd, last_batch=bi)

    fold_probs  = np.array(fp)
    fold_labels = np.array(fl)
    np.savez(fold_done, probs=fold_probs, labels=fold_labels, ids=fi, datasets=fd)
    if partial.exists():
        partial.unlink()

    all_probs.extend(fold_probs.tolist())
    all_labels.extend(fold_labels.tolist())
    all_ids.extend(fi)
    all_datasets.extend(fd)
    all_folds_list.extend([fold] * len(fold_labels))

    elapsed = time.time() - fold_start
    print(f'  {len(fold_labels):,} samples | {int(fold_labels.sum())} pos | {elapsed/60:.1f} min')
    if device == 'cuda':
        torch.cuda.empty_cache()

all_probs     = np.array(all_probs)
all_labels    = np.array(all_labels)
all_folds_arr = np.array(all_folds_list)

# Pre-flight check
assert not np.any(np.isnan(all_probs)),  'NaN in predictions — check model weights'
assert not np.any(np.isnan(all_labels)), 'NaN in labels'
assert set(np.unique(all_labels)) <= {0, 1}, 'Labels must be binary'

print(f'\nInference complete: {len(all_labels):,} samples | '
      f'{int(all_labels.sum())} pos ({100*all_labels.mean():.2f}%) | '
      f'{(time.time()-total_start)/60:.1f} min total')


## Cell 5 — Primary Metrics (Official PhysioNet)

In [ ]:
np.random.seed(SEED)

if OFFICIAL:
    tpr_5pct = float(compute_challenge_score(
        all_labels.astype(np.float64), all_probs.astype(np.float64),
        fraction_capacity=0.05, num_permutations=N_PERMS_FINAL, seed=SEED,
    ))
    auroc_val, auprc_val = _compute_auc(all_labels, all_probs)
    auroc = float(auroc_val)
    auprc = float(auprc_val)
else:
    fpr_, tpr_, _ = roc_curve(all_labels, all_probs)
    idx5 = np.where(fpr_ <= 0.05)[0]
    tpr_5pct = float(tpr_[idx5[-1]]) if len(idx5) > 0 else 0.0
    auroc = float(roc_auc_score(all_labels, all_probs))
    auprc = float(average_precision_score(all_labels, all_probs))

method_tag = f'OFFICIAL {N_PERMS_FINAL} perms' if OFFICIAL else 'sklearn approx'
print(f'TPR@5%:  {tpr_5pct:.4f}  [{method_tag}]')
print(f'AUROC:   {auroc:.4f}')
print(f'AUPRC:   {auprc:.4f}')

# ── Benchmark comparison ──────────────────────────────────────────────────
print('\nBenchmark comparison:')
benchmarks = [
    ('Random baseline',                        0.050),
    ('No-pretrain baseline (expected)',         0.300),
    ('Challenge target (beat PhysioNet score)', 0.420),
    ('Kim et al. 2025 (2D approach)',           0.369),
    ('Van Santvliet 2025 top team (val set)',   0.445),
    ('Van Santvliet 2025 CV mean',              0.490),
]
for name, val in benchmarks:
    diff = tpr_5pct - val
    mark = '↑' if diff >= 0 else '↓'
    print(f'  {mark}{abs(diff):.4f}  vs  {name} ({val:.3f})')

# ── Clinical interpretation ───────────────────────────────────────────────
N_total  = len(all_labels)
n_pos    = int(all_labels.sum())
capacity = int(0.05 * N_total)
found    = int(tpr_5pct * n_pos)
random_f = max(1, int(0.05 * n_pos))
nns      = round(capacity / found, 1) if found > 0 else float('inf')

print(f'\nClinical interpretation:')
print(f'  Screening capacity (5%):  {capacity:,} patients')
print(f'  Chagas cases found:       {found} / {n_pos} ({100*tpr_5pct:.1f}%)')
print(f'  Improvement over random:  {found/random_f:.1f}×')
print(f'  NNS (Number Needed to Screen to find 1 case): {nns}')
print(f'  (Random NNS: {round(capacity/random_f, 1)})')


## Cell 6 — Threshold-Based Metrics

In [ ]:
fpr_arr, tpr_arr, roc_thr = roc_curve(all_labels, all_probs)
prec_arr, rec_arr, pr_thr  = precision_recall_curve(all_labels, all_probs)

# Optimal Youden J  (maximises sensitivity + specificity)
j_idx      = np.argmax(tpr_arr - fpr_arr)
thr_youden = float(roc_thr[j_idx])

# Optimal F1
f1_arr   = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
f1_idx   = np.argmax(f1_arr)
thr_f1   = float(pr_thr[f1_idx])

results_thr = {}
for name, thr in [('default_0.5', 0.5), ('youden_j', thr_youden), ('optimal_f1', thr_f1)]:
    pred = (all_probs >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(all_labels, pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    results_thr[name] = dict(
        threshold   = round(thr, 4),
        TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
        sensitivity = round(recall_score(all_labels, pred), 4),
        specificity = round(spec, 4),
        precision   = round(precision_score(all_labels, pred, zero_division=0), 4),
        npv         = round(npv, 4),
        f1          = round(f1_score(all_labels, pred, zero_division=0), 4),
        mcc         = round(float(matthews_corrcoef(all_labels, pred)), 4),
        accuracy    = round(float(accuracy_score(all_labels, pred)), 4),
    )

df_thr = pd.DataFrame(results_thr).T
cols   = ['threshold','sensitivity','specificity','precision','npv','f1','mcc','accuracy']
print('Threshold analysis:')
print(df_thr[cols].to_string())

# Primary threshold for thesis = Youden J (screening focus: maximise sensitivity)
primary = results_thr['youden_j']
print(f'\nPrimary (Youden J, thr={primary["threshold"]}):')
for k in ['sensitivity','specificity','precision','npv','f1','mcc','accuracy']:
    print(f'  {k:<14}: {primary[k]:.4f}')
print(f'  Confusion  TP={primary["TP"]}  TN={primary["TN"]:,}  FP={primary["FP"]:,}  FN={primary["FN"]}')


## Cell 7 — Bootstrap 95% Confidence Intervals

In [ ]:
np.random.seed(SEED)
n = len(all_labels)
bt_tpr, bt_auroc, bt_auprc = [], [], []

for _ in tqdm(range(N_BOOTSTRAP), desc='Bootstrap', leave=False):
    idx  = np.random.choice(n, n, replace=True)
    lbl  = all_labels[idx]
    prb  = all_probs[idx]
    if lbl.sum() < 2 or (lbl == 0).sum() < 2:
        continue
    # TPR@5% approximation via ROC (fast; no permutation loop in bootstrap)
    fpr_b, tpr_b, _ = roc_curve(lbl, prb)
    i5 = np.where(fpr_b <= 0.05)[0]
    bt_tpr.append(float(tpr_b[i5[-1]]) if len(i5) > 0 else 0.0)
    bt_auroc.append(float(roc_auc_score(lbl, prb)))
    bt_auprc.append(float(average_precision_score(lbl, prb)))

def ci95(arr):
    a = np.array(arr)
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

tpr_lo,   tpr_hi   = ci95(bt_tpr)
auroc_lo, auroc_hi = ci95(bt_auroc)
auprc_lo, auprc_hi = ci95(bt_auprc)

print(f'95% bootstrap CI ({N_BOOTSTRAP} resamples):')
print(f'  TPR@5%:  {tpr_5pct:.4f}  [{tpr_lo:.4f}, {tpr_hi:.4f}]')
print(f'  AUROC:   {auroc:.4f}  [{auroc_lo:.4f}, {auroc_hi:.4f}]')
print(f'  AUPRC:   {auprc:.4f}  [{auprc_lo:.4f}, {auprc_hi:.4f}]')
print()
print('Note: TPR@5% CI uses simplified ROC bootstrap (not full permutation-based).')
print('Report as approximate; exact permutation CI requires 10k×1k loops.')


## Cell 8 — Per-Dataset Analysis

In [ ]:
ds_rows = []
for ds_name in ['ptbxl', 'samitrop', 'code15']:
    mask = np.array([d == ds_name for d in all_datasets])
    if not mask.any():
        continue
    dl, dp = all_labels[mask], all_probs[mask]
    row = dict(dataset=ds_name.upper(), n_total=int(mask.sum()), n_pos=int(dl.sum()))

    unique_cls = np.unique(dl)
    if len(unique_cls) < 2:
        # Single class — AUROC and TPR@5% are undefined
        row.update(tpr_5pct='n/a (single class)', auroc='n/a', auprc='n/a')
        note = f'all {"positive" if dl.mean()==1 else "negative"}'
        print(f'{ds_name.upper()}: {row["n_total"]:,} samples — {note}, metrics undefined')
    else:
        if OFFICIAL:
            ds_tpr = float(compute_challenge_score(
                dl.astype(np.float64), dp.astype(np.float64),
                fraction_capacity=0.05, num_permutations=N_PERMS_FOLD, seed=SEED,
            ))
            ds_auroc, ds_auprc = _compute_auc(dl, dp)
        else:
            fpr_d, tpr_d, _ = roc_curve(dl, dp)
            i5d = np.where(fpr_d <= 0.05)[0]
            ds_tpr   = float(tpr_d[i5d[-1]]) if len(i5d) > 0 else 0.0
            ds_auroc = float(roc_auc_score(dl, dp))
            ds_auprc = float(average_precision_score(dl, dp))
        row.update(tpr_5pct=round(ds_tpr, 4),
                   auroc=round(float(ds_auroc), 4),
                   auprc=round(float(ds_auprc), 4))
        print(f'{ds_name.upper()}: {row["n_total"]:,} samples | '
              f'TPR@5%={ds_tpr:.4f}  AUROC={float(ds_auroc):.4f}  AUPRC={float(ds_auprc):.4f}')
    ds_rows.append(row)

df_ds = pd.DataFrame(ds_rows)
df_ds.to_csv(CHECKPOINT_DIR / 'per_dataset_metrics.csv', index=False)


## Cell 9 — Per-Fold Performance Table

In [ ]:
fold_rows = []
for fold in range(5):
    mask = all_folds_arr == fold
    fl, fp2 = all_labels[mask], all_probs[mask]
    row = dict(fold=fold, n_total=int(mask.sum()), n_pos=int(fl.sum()))

    if len(np.unique(fl)) < 2:
        row.update(tpr_5pct='n/a', auroc='n/a', auprc='n/a')
    else:
        if OFFICIAL:
            ft = float(compute_challenge_score(
                fl.astype(np.float64), fp2.astype(np.float64),
                fraction_capacity=0.05, num_permutations=N_PERMS_FOLD, seed=SEED,
            ))
            fa, fp3 = _compute_auc(fl, fp2)
        else:
            fpr_f, tpr_f, _ = roc_curve(fl, fp2)
            i5f = np.where(fpr_f <= 0.05)[0]
            ft  = float(tpr_f[i5f[-1]]) if len(i5f) > 0 else 0.0
            fa  = float(roc_auc_score(fl, fp2))
            fp3 = float(average_precision_score(fl, fp2))
        row.update(tpr_5pct=round(ft, 4),
                   auroc=round(float(fa), 4),
                   auprc=round(float(fp3), 4))
    fold_rows.append(row)

# Ensemble row
fold_rows.append(dict(fold='Ensemble',
                      n_total=len(all_labels), n_pos=int(all_labels.sum()),
                      tpr_5pct=round(tpr_5pct, 4),
                      auroc=round(auroc, 4),
                      auprc=round(auprc, 4)))

df_folds = pd.DataFrame(fold_rows)
df_folds.to_csv(CHECKPOINT_DIR / 'per_fold_metrics.csv', index=False)
print(df_folds.to_string(index=False))

numeric_tpr = [r['tpr_5pct'] for r in fold_rows[:5] if isinstance(r['tpr_5pct'], float)]
if len(numeric_tpr) == 5:
    print(f'\nFold mean ± std:  {np.mean(numeric_tpr):.4f} ± {np.std(numeric_tpr):.4f}')
    print(f'Ensemble gain:    +{tpr_5pct - np.mean(numeric_tpr):.4f}')
    print(f'Van Santvliet CV: 0.490 ± 0.008  (for comparison)')


## Cell 10 — Thesis Figures (300 dpi)

In [ ]:
def save_fig(name):
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

thr = primary['threshold']
pred_binary = (all_probs >= thr).astype(int)

# ── Figure 4.1: ROC Curve ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_arr, tpr_arr, lw=2.5, color='#2E86AB',
        label=f'ChagaSight ensemble  AUC = {auroc:.3f}  [{auroc_lo:.3f}–{auroc_hi:.3f}]')
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.5, label='Random classifier')
i5 = np.argmin(np.abs(fpr_arr - 0.05))
ax.plot(fpr_arr[i5], tpr_arr[i5], 'ro', ms=10, zorder=5,
        label=f'5% FPR  TPR = {tpr_arr[i5]:.3f}')
ax.axvline(0.05, color='grey', ls=':', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Figure 4.1 — ROC Curve')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
save_fig('fig4_1_roc_curve.png')

# ── Figure 4.2: Precision-Recall Curve ───────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(rec_arr, prec_arr, lw=2.5, color='#A23B72',
        label=f'ChagaSight ensemble  AP = {auprc:.3f}  [{auprc_lo:.3f}–{auprc_hi:.3f}]')
baseline_prec = all_labels.mean()
ax.axhline(baseline_prec, color='k', ls='--', lw=1.2, alpha=0.5,
           label=f'Random classifier  ({baseline_prec:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Figure 4.2 — Precision-Recall Curve')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
save_fig('fig4_2_pr_curve.png')

# ── Figure 4.3: Confusion Matrix ─────────────────────────────────────────
cm = confusion_matrix(all_labels, pred_binary)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Neg', 'Predicted Pos'],
            yticklabels=['True Neg', 'True Pos'],
            annot_kws={'size': 14, 'weight': 'bold'}, ax=ax,
            cbar_kws={'label': 'Count'})
total_cm = cm.sum()
for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.72, f'({100*cm[i,j]/total_cm:.1f}%)',
                ha='center', va='center', fontsize=10, color='dimgrey')
ax.set_title(f'Figure 4.3 — Confusion Matrix  (threshold = {thr:.4f}, Youden J)')
plt.tight_layout()
save_fig('fig4_3_confusion_matrix.png')

# ── Figure 4.4: Probability Histogram by Class ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(all_probs[all_labels == 0], bins=60, alpha=0.6, density=True,
        color='steelblue', label=f'Negative  n={int((all_labels==0).sum()):,}')
ax.hist(all_probs[all_labels == 1], bins=60, alpha=0.6, density=True,
        color='crimson',   label=f'Positive  n={int(all_labels.sum()):,}')
ax.axvline(thr, color='k', ls='--', lw=1.5, label=f'Threshold {thr:.3f}')
ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Density')
ax.set_title('Figure 4.4 — Predicted Probability by Class')
ax.legend()
plt.tight_layout()
save_fig('fig4_4_prob_histogram.png')

# ── Figure 4.5: Calibration Reliability Diagram ──────────────────────────
n_bins    = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_mids  = (bin_edges[:-1] + bin_edges[1:]) / 2
frac_pos  = np.zeros(n_bins); mean_pred = np.zeros(n_bins); bin_cts = np.zeros(n_bins)
for i in range(n_bins):
    mask_b = (all_probs >= bin_edges[i]) & (all_probs <= bin_edges[i+1])
    if mask_b.sum() > 0:
        frac_pos[i]  = all_labels[mask_b].mean()
        mean_pred[i] = all_probs[mask_b].mean()
        bin_cts[i]   = mask_b.sum()
valid_b = bin_cts > 0
fig, (ax_c, ax_h) = plt.subplots(2, 1, figsize=(7, 8),
                                   gridspec_kw={'height_ratios': [3, 1]})
ax_c.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Perfect calibration')
ax_c.plot(mean_pred[valid_b], frac_pos[valid_b], 'o-', lw=2, ms=7,
          color='#E07B39', label='ChagaSight ensemble')
ax_c.set_xlabel('Mean Predicted Probability'); ax_c.set_ylabel('Fraction of Positives')
ax_c.set_title('Figure 4.5 — Calibration Reliability Diagram')
ax_c.legend(); ax_c.set_xlim(-0.02, 1.02); ax_c.set_ylim(-0.02, 1.02)
ax_h.bar(bin_mids, bin_cts, width=0.08, color='steelblue', alpha=0.7)
ax_h.set_xlabel('Predicted Probability'); ax_h.set_ylabel('Count')
plt.tight_layout()
save_fig('fig4_5_calibration.png')

# ── Figure 4.6: Per-Fold and Ensemble Bar Chart ──────────────────────────
fold_tpr_vals  = [r['tpr_5pct'] if isinstance(r['tpr_5pct'], float) else 0 for r in fold_rows]
fold_bar_lbls  = [f'Fold {r["fold"]}' if r['fold'] != 'Ensemble' else 'Ensemble'
                  for r in fold_rows]
bar_colors     = ['#4472C4'] * 5 + ['#ED7D31']
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(len(fold_bar_lbls)), fold_tpr_vals, color=bar_colors,
              edgecolor='black', linewidth=0.7, alpha=0.88)
ax.axhline(0.369, color='dodgerblue', ls='-.', lw=1.5, alpha=0.8,
           label='Kim 2025 val set (0.369)')
ax.axhline(0.420, color='green',      ls='--', lw=1.5, alpha=0.8,
           label='Challenge target (0.420)')
ax.axhline(0.445, color='red',        ls='--', lw=1.5, alpha=0.8,
           label='Van Santvliet val (0.445)')
ax.axhline(0.490, color='purple',     ls=':',  lw=1.5, alpha=0.8,
           label='Van Santvliet CV (0.490)')
for bar, val in zip(bars, fold_tpr_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.004,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_xticks(range(len(fold_bar_lbls))); ax.set_xticklabels(fold_bar_lbls)
ax.set_ylabel('TPR @ 5% FPR'); ax.set_title('Figure 4.6 — Per-Fold and Ensemble Performance')
ax.legend(fontsize=8, loc='lower right')
ax.set_ylim(0, min(1.0, max(fold_tpr_vals) * 1.2))
plt.tight_layout()
save_fig('fig4_6_per_fold_bar.png')

print(f'\n6 figures saved to: {FIGURES_DIR}')


## Cell 11 — Summary Table (Thesis-Ready)

In [ ]:
summary = {
    'TPR @ 5% FPR (primary)':  f'{tpr_5pct:.4f}  [{tpr_lo:.4f}–{tpr_hi:.4f}]',
    'AUROC':                    f'{auroc:.4f}  [{auroc_lo:.4f}–{auroc_hi:.4f}]',
    'AUPRC':                    f'{auprc:.4f}  [{auprc_lo:.4f}–{auprc_hi:.4f}]',
    'Sensitivity (recall)':     f'{primary["sensitivity"]:.4f}',
    'Specificity':              f'{primary["specificity"]:.4f}',
    'Precision (PPV)':          f'{primary["precision"]:.4f}',
    'NPV':                      f'{primary["npv"]:.4f}',
    'F1 Score':                 f'{primary["f1"]:.4f}',
    'MCC':                      f'{primary["mcc"]:.4f}',
    'Accuracy':                 f'{primary["accuracy"]:.4f}',
    'Optimal threshold':        f'{primary["threshold"]:.4f}  (Youden J)',
    'TP / TN / FP / FN':       f'{primary["TP"]} / {primary["TN"]:,} / {primary["FP"]:,} / {primary["FN"]}',
    'Number Needed to Screen':  str(nns),
    'Total samples':            f'{len(all_labels):,}',
    'Positive samples':         f'{int(all_labels.sum()):,}  ({100*all_labels.mean():.2f}%)',
    'Ensemble models':          '5 (5-fold CV)',
    'Params per model':         f'{total_params:,}',
    'Bootstrap CI resamples':   f'{N_BOOTSTRAP}',
    'Primary metric method':    'OFFICIAL PhysioNet (helper_code.py)' if OFFICIAL else 'sklearn approx',
    '--- Comparison ---':       '',
    'vs Kim 2025 val set':      f'{tpr_5pct - 0.369:+.4f}  (Kim: 0.369)',
    'vs Van Santvliet val set': f'{tpr_5pct - 0.445:+.4f}  (VS: 0.445)',
    'vs Van Santvliet CV mean': f'{tpr_5pct - 0.490:+.4f}  (VS: 0.490)',
}

df_summary = pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])
df_summary.index.name = 'Metric'
print(df_summary.to_string())

# Save all CSV outputs
df_summary.to_csv(CHECKPOINT_DIR / 'ensemble_summary.csv')
df_thr.to_csv(CHECKPOINT_DIR / 'threshold_comparison.csv')
pd.DataFrame({
    'id': all_ids, 'fold': all_folds_arr, 'dataset': all_datasets,
    'true_label': all_labels, 'predicted_probability': all_probs,
    'predicted_class': pred_binary,
}).to_csv(CHECKPOINT_DIR / 'ensemble_predictions.csv', index=False)

print('\nFiles saved:')
for f in ['ensemble_summary.csv', 'threshold_comparison.csv',
          'per_dataset_metrics.csv', 'per_fold_metrics.csv',
          'ensemble_predictions.csv']:
    print(f'  checkpoints/{f}')
n_figs = len(list(FIGURES_DIR.glob('*.png')))
print(f'  thesis_figures/  ({n_figs} PNG files @ 300 dpi)')


## Cell 12 — Package Final Ensemble Model

In [ ]:
pkg = {
    'model_config': dict(
        img_size=(24, 2048), patch_size_2d=(8, 64),
        num_leads=12, seq_len_1d=1000, patch_size_1d=50,
        embed_dim=768, depth=12, num_heads=12,
        use_aol=True, use_demographics=True,
    ),
    'ensemble_metrics': {
        'tpr_5pct': tpr_5pct, 'auroc': auroc, 'auprc': auprc,
        'tpr_ci': (tpr_lo, tpr_hi),
        'auroc_ci': (auroc_lo, auroc_hi),
        'threshold': primary['threshold'],
        'n_total': len(all_labels), 'n_positive': int(all_labels.sum()),
        'official': OFFICIAL,
    },
    'fold_val_scores': fold_val_scores,
    'fold_models': [],
}
for fold, ckpt_path in enumerate(fold_ckpts):
    c = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    pkg['fold_models'].append({
        'fold': fold,
        'model_state_dict': c['model_state_dict'],
        'val_score': c.get('val_score', None),
    })

out_path = CHECKPOINT_DIR / 'FINAL_ENSEMBLE_MODEL.pt'
torch.save(pkg, out_path)
mb = out_path.stat().st_size / 1e6
print(f'Saved: FINAL_ENSEMBLE_MODEL.pt  ({mb:.0f} MB)')
print('Contains: 5 fold weights + metrics (with CI) + model config')
